# Parte 1: DCGAN de dragones de cuerpo completo

Universo de fantasía medieval con alas, escamas y cuernos. Los personajes finales deben ser nuevos.

Modelo, entrenamiento y generación implementados; dataset real y resultados pendientes. Las pruebas sintéticas son técnicas, no evidencias del proyecto. Por defecto este notebook no inicia entrenamiento ni generación. Trabajar desde main; no hacer commits automáticamente.

## 1. Localizar el repositorio y cargar configuración

Ejecutar con el entorno `.venv` descrito en `docs/trabajo_local.md`. Mantener la misma configuración para las tres comparaciones. Los parámetros iniciales requieren un piloto antes de fijar el presupuesto.

In [ ]:
import json
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents) if (p / "configs/base.json").is_file() and (p / "dragon_gan").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Abrir el notebook dentro del repositorio local.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
CONFIG = json.loads((ROOT / "configs/base.json").read_text(encoding="utf-8"))
print(json.dumps(CONFIG, indent=2))

## 2. Arquitectura y entorno

G transforma un vector z en RGB de 64×64 usando convoluciones transpuestas y salida Tanh. D usa convoluciones y devuelve logits. La carga mantiene proporción y rellena con blanco, sin recortar alas. Acordar estas transformaciones con persona 2.

Referencia de arquitectura: [tutorial DCGAN de PyTorch](https://docs.pytorch.org/tutorials/beginner/dcgan_faces_tutorial.html).

In [ ]:
import torch
from dragon_gan.models import build_models
from dragon_gan.train import seed_everything

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
seed_everything(CONFIG["seed"], CONFIG["threads"])
generator, discriminator = build_models(CONFIG, torch.device(DEVICE))
print("Dispositivo:", DEVICE, "| PyTorch:", torch.__version__)
print("Parámetros G:", sum(p.numel() for p in generator.parameters()))
print("Parámetros D:", sum(p.numel() for p in discriminator.parameters()))
# No se ha entrenado: no usar las salidas iniciales como galería final.

## 3. Hipótesis antes del entrenamiento

Cada ejecución guarda su hipótesis antes de entrenar. Revisarlas y adaptar el criterio de éxito antes de activar una ejecución. Se cambia una sola cosa respecto a base; no encadenar los pesos de una variante a otra.

In [ ]:
from dragon_gan.train import HYPOTHESES, train

for name, hypothesis in HYPOTHESES.items():
    print(name + ": " + hypothesis + "\n")

## 4. Dataset y piloto

Persona 2 entregará las imágenes y su documentación de origen/permiso. Colocarlas en `data/dragons`. El hash automático no verifica permisos. Usar un piloto de una época para estimar el tiempo real; la prueba sintética no permite estimar duración con dragones.

In [ ]:
DATASET = ROOT / "data/dragons"
if not DATASET.is_dir():
    print("PENDIENTE: dataset real de persona 2.")
else:
    from dragon_gan.data import DragonDataset
    dataset = DragonDataset(DATASET, CONFIG["image_size"])
    print("Imágenes disponibles:", len(dataset))

## 5. Ejecutar una configuración

Para el piloto, activar `RUN_TRAINING`, elegir `RUN_NAME="pilot"` y `STOP_AFTER=1`. Después fijar el presupuesto en configs/base.json, recargar CONFIG y comenzar ejecuciones nuevas con `STOP_AFTER=None`.

Comparaciones: base, loss y stabilization, siempre con los mismos datos y parámetros comunes. Usar carpetas separadas. Para reanudar, conservar nombre y configuración y poner `RESUME=True`; se recupera la última época completa.

In [ ]:
RUN_TRAINING = False
EXPERIMENT = "base"  # base, loss, stabilization
RUN_NAME = "pilot"
STOP_AFTER = 1  # None para completar todas las épocas acordadas
RESUME = False
OUTPUT = ROOT / "runs" / RUN_NAME

if RUN_TRAINING:
    checkpoint = train(CONFIG, DATASET, OUTPUT, experiment=EXPERIMENT, device=DEVICE,
                       resume=OUTPUT / "last.pt" if RESUME else None, stop_after=STOP_AFTER)
    print("Checkpoint:", checkpoint)
else:
    print("Entrenamiento desactivado. Revisar datos, hipótesis y parámetros antes de activarlo.")

## 6. Análisis que debes escribir

Consultar losses.csv, losses.png y samples de cada ejecución. Comparar evolución con el mismo ruido fijo, reconocimiento, diversidad y fallos. Explicar las curvas; una pérdida de D cercana a cero no basta para concluir éxito. Las pérdidas de G de minimax y no saturante tienen significados diferentes. Registrar resultados negativos y limitaciones.

Persona 2 recibirá las figuras, registros, configuración y pesos para elaborar vecinos más cercanos y la presentación.

## 7. Generar candidatos con el modelo elegido

Solo después de entrenar y elegir pesos reales. El manifiesto conserva vectores z, semillas y cantidad generada. Los candidatos aún necesitan selección y prueba de vecinos. Ver docs/entrenamiento.md para regenerar la galería seleccionada.

In [ ]:
from dragon_gan.generate import generate

RUN_GENERATION = False
CHOSEN_CHECKPOINT = ROOT / "runs/base/last.pt"
CANDIDATES = ROOT / "runs/candidates"
if RUN_GENERATION:
    generate(CHOSEN_CHECKPOINT, CANDIDATES, count=200, seed=1000)
else:
    print("Generación desactivada: falta elegir un modelo entrenado con datos reales.")

## Resultados y limitaciones

Pendiente: entrenamiento real, interpretación de pérdidas, comparación de experimentos, galería, vecinos y reflexión sobre el dragón más difícil de defender como nuevo. No sustituir estos resultados con imágenes sintéticas de prueba.